# 8 - GRACE Joint Inversion

This notebook extends the joint inversion framework by adding **GRACE**
gravity observations as a third data source alongside ocean SSH altimetry
and ice-surface altimetry.

GRACE measures changes in Earth's gravity field caused by redistributions
of mass — in particular, ice and firn loss. Because gravity responds to
the *total* mass change (ice + firn combined), it provides complementary
information to altimetry, which measures *volume* change and therefore
cannot disentangle ice from firn without prior assumptions on density.

The notebook is split into two main parts:

1. **Full inversion with GRACE** — run the preconditioned CG Bayesian
   inversion with all three data sources and inspect the posterior
   component maps (ice thickness, firn thickness, ocean dynamics) and
   the joint ice/firn GMSL covariance.

2. **Knockout test** — run four variants of the inversion (full,
   no-SSH, no-ice-altimetry, no-GRACE) to assess the individual
   contribution of each data type to GMSL uncertainty.

## Part 1: Full GRACE Joint Inversion

### Setup

Import all required libraries and set global parameters.

Key parameters:
- `lmax = 128` — spherical harmonic truncation for the full-resolution model.
- `measure_error_std = 0.001` — altimetry noise standard deviation (non-dimensionalised).
- `grace_std_dev_m = 0.0027` — GRACE noise in metres equivalent water height.
- `grace_observation_degree = 96` — maximum spherical harmonic degree observed by GRACE.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tqdm import tqdm
import cartopy.crs as ccrs
from pygeoinf import (
    BlockDiagonalLinearOperator, BlockLinearOperator, CGMatrixSolver,
    EigenSolver, EuclideanSpace, GaussianMeasure, HilbertSpaceDirectSum,
    LinearBayesianInversion, LinearForwardProblem, RowLinearOperator,
)
from pyslfp import (
    FingerPrint, IceModel, plot,
    sea_level_change_to_load_operator, sea_surface_height_operator,
)
from pyslfp.operators import grace_operator
from project import colors
from pygeoinf_extras import standard_dev
from pygeoinf_extras.plots import plot_bivariate_corner
from pyslfp_extras.altimetry import GridPoints
from pyslfp_extras.ice_thickness import IceSheetChange
from pyslfp_extras.ocean_dynamics import OceanDynamics

NOTEBOOK_DIR = Path.cwd()
FIGURES_DIR = NOTEBOOK_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

measure_error_std        = 0.001
grace_std_dev_m          = 0.0027
grace_observation_degree = 96

### Full-resolution model setup

Build the `FingerPrint` model at `lmax = 128` and construct the three
component models:

- **`IceSheetChange`** — ice and firn thickness changes, including their
  mass-load operators and prior Gaussian measures. The firn density is
  set to 30% of ice density, consistent with densification assumptions.
- **`OceanDynamics`** — steric and dynamic ocean topography (ODT) changes,
  modelled as a Gaussian random field over the ocean surface.

The model space is the direct sum of the three component spaces and the
prior is similarly a product Gaussian measure.

In [ ]:
fp = FingerPrint(lmax=128)
fp.set_state_from_ice_ng(version=IceModel.ICE7G, date=0.0)
fp_op = fp.as_sobolev_linear_operator(2, fp.mean_sea_floor_radius * 0.1)

grace_std = grace_std_dev_m / fp.length_scale

ice = IceSheetChange.global_ice(
    finger_print=fp,
    finger_print_operator=fp_op,
    length_scale=0.1 * fp.mean_sea_floor_radius,
    pattern=IceSheetChange.ThicknessWeightedPattern(),
    ice_gmsl_std=0.003,
    firn_gmsl_std=0.002,
    firn_density=0.3 * fp.ice_density,
    include_firn=True,
)

OD_pattern = OceanDynamics.DataPattern()
odt = OceanDynamics(
    finger_print=fp,
    finger_print_operator=fp_op,
    std=0.004,
    length_scale=10000.0,
    pattern=OD_pattern,
)

model_space = HilbertSpaceDirectSum([
    ice.ice_thickness.domain,
    ice.firn_thickness.domain,
    odt.height_measure.domain,
])
model_prior = GaussianMeasure.from_direct_sum([
    ice.ice_thickness,
    ice.firn_thickness,
    odt.height_measure,
])

print('Full-resolution model built.')

### Observation geometry

Place altimetry observation points on a regular grid:
- **SSH altimetry** — ocean grid at 10° spacing, limited to latitudes
  below 66° (typical of conventional altimetry swath coverage).
- **Ice altimetry** — ice-sheet grid at 15° spacing.

GRACE observations are in spectral space (spherical harmonic coefficients
up to degree 96) and do not require a spatial grid.

In [ ]:
ssh_altimetry = GridPoints.ocean_altimetry(fp, 10.0, 66.0)
ice_altimetry = GridPoints.ice(fp, 15.0)

print(f'SSH altimetry points : {len(ssh_altimetry.coords)}')
print(f'Ice altimetry points : {len(ice_altimetry.coords)}')
print(f'GRACE SH coefficients: degrees 0-{grace_observation_degree}')

### Forward operator with GRACE

The forward operator is factored into three matrices:

$$G = P_{\mathrm{left}} \; F_{\mathrm{middle}} \; L_{\mathrm{right}}$$

- $L_{\mathrm{right}}$ **(4x3)** routes the three model components
  (ice thickness, firn thickness, ODT height) into a 4-component
  intermediate space (load, ODT pass-through, ice pass-through, firn pass-through).
- $F_{\mathrm{middle}}$ **(4x4, block-diagonal)** applies the fingerprint
  operator to the load component; identity operators to the other three.
- $P_{\mathrm{left}}$ **(3x4)** extracts the three observation types:
  - **Row 0 (SSH altimetry):** applies the SSH operator then samples at ocean points.
  - **Row 1 (ice altimetry):** samples ice and firn fields at ice-surface points.
  - **Row 2 (GRACE):** applies the spherical harmonic sampling operator to the fingerprint response.

This structure means the expensive fingerprint computation is shared across all three observation types.

In [ ]:
def build_forward_operator(fp, fp_op, ice, odt,
                           ssh_altimetry, ice_altimetry,
                           grace_observation_degree):
    load_space     = fp_op.domain
    response_space = fp_op.codomain
    ice_space      = ice.ice_thickness.domain
    firn_space     = ice.firn_thickness.domain
    odt_space      = odt.height_measure.domain

    F   = fp_op
    S   = sea_surface_height_operator(fp, response_space)
    L_I = ice.ice_thickness_to_load_operator
    L_F = ice.firn_thickness_to_load_operator
    L_W = sea_level_change_to_load_operator(fp, load_space)

    P_S_ssh  = ssh_altimetry.point_evaluation_operator(S.codomain)
    P_S_odt  = ssh_altimetry.point_evaluation_operator(odt_space)
    P_I_ice  = ice_altimetry.point_evaluation_operator(ice_space)
    P_I_firn = ice_altimetry.point_evaluation_operator(firn_space)
    grace_op = grace_operator(response_space, grace_observation_degree)

    id_odt  = odt_space.identity_operator()
    id_ice  = ice_space.identity_operator()
    id_firn = firn_space.identity_operator()

    ssh_obs   = P_S_ssh.codomain
    ice_obs   = P_I_ice.codomain
    grace_obs = grace_op.codomain

    # L_right: routes model components to intermediate spaces
    L_right = BlockLinearOperator([
        [L_I, L_F, L_W],
        [ice_space.zero_operator(codomain=odt_space),
         firn_space.zero_operator(codomain=odt_space), id_odt],
        [id_ice, firn_space.zero_operator(codomain=ice_space),
         odt_space.zero_operator(codomain=ice_space)],
        [ice_space.zero_operator(codomain=firn_space), id_firn,
         odt_space.zero_operator(codomain=firn_space)],
    ])
    # F_middle: fingerprint on load, identity on everything else
    F_middle = BlockDiagonalLinearOperator([F, id_odt, id_ice, id_firn])
    # P_left: one row per observation type
    P_left = BlockLinearOperator([
        [P_S_ssh @ S, P_S_odt,
         ice_space.zero_operator(codomain=ssh_obs),
         firn_space.zero_operator(codomain=ssh_obs)],
        [response_space.zero_operator(codomain=ice_obs),
         odt_space.zero_operator(codomain=ice_obs), P_I_ice, P_I_firn],
        [grace_op, odt_space.zero_operator(codomain=grace_obs),
         ice_space.zero_operator(codomain=grace_obs),
         firn_space.zero_operator(codomain=grace_obs)],
    ])
    return P_left @ F_middle @ L_right, grace_op.codomain


forward_operator, grace_obs = build_forward_operator(
    fp, fp_op, ice, odt, ssh_altimetry, ice_altimetry, grace_observation_degree,
)
data_space = forward_operator.codomain

model_space_to_slc_operator = RowLinearOperator([
    ice.load_to_slc_operator @ ice.ice_thickness_to_load_operator,
    ice.load_to_slc_operator @ ice.firn_thickness_to_load_operator,
    odt._height_to_slc_op,
])

print('Forward operator built.')
print(f'Data space dimension: {data_space.dimension}')

### Data error model and synthetic data

Each observation type has its own noise level:
- SSH and ice altimetry share `measure_error_std` (white noise).
- GRACE uses `grace_std` (converted to non-dimensional units).

A synthetic true model is drawn from the prior, and noisy observations
are generated by applying the forward operator and adding noise samples.

In [ ]:
ssh_obs_space = data_space.subspaces[0]
ice_obs_space = data_space.subspaces[1]

data_error_measure = GaussianMeasure.from_direct_sum([
    GaussianMeasure.from_standard_deviation(ssh_obs_space, measure_error_std),
    GaussianMeasure.from_standard_deviation(ice_obs_space, measure_error_std),
    GaussianMeasure.from_standard_deviation(grace_obs,     grace_std),
])

forward_problem = LinearForwardProblem(
    forward_operator, data_error_measure=data_error_measure,
)
model_true, data = forward_problem.synthetic_model_and_data(model_prior)

print('Synthetic true model and data generated.')

### Preconditioner setup

A preconditioner dramatically accelerates the conjugate gradient (CG)
solver by providing a cheap approximation to the inverse of the normal
operator.

We build a **low-resolution** version of the same problem at `lmax = 32`.
The low-res model has far fewer degrees of freedom, so its normal operator
can be inverted exactly via eigen-decomposition (`EigenSolver`). This
inverse is then used as the preconditioner for the full-resolution CG solve.

Importantly, the low-res preconditioner samples at the **same geographic
coordinates** as the full-resolution model so the data spaces are compatible.

In [ ]:
lmax_precon = 32

precon_fp = FingerPrint(lmax=lmax_precon)
precon_fp.set_state_from_ice_ng(version=IceModel.ICE7G, date=0.0)
precon_fp_op = precon_fp.as_sobolev_linear_operator(
    2, precon_fp.mean_sea_floor_radius * 0.1
)

precon_ice = IceSheetChange.global_ice(
    finger_print=precon_fp, finger_print_operator=precon_fp_op,
    length_scale=0.1 * precon_fp.mean_sea_floor_radius,
    pattern=IceSheetChange.ThicknessWeightedPattern(),
    ice_gmsl_std=0.003, include_firn=True,
)
precon_odt = OceanDynamics(
    finger_print=precon_fp, finger_print_operator=precon_fp_op,
    std=0.004, length_scale=10000.0, pattern=OD_pattern,
)
precon_model_prior = GaussianMeasure.from_direct_sum([
    precon_ice.ice_thickness,
    precon_ice.firn_thickness,
    precon_odt.height_measure,
])

precon_response_space = precon_fp_op.codomain
precon_ice_space      = precon_ice.ice_thickness.domain
precon_firn_space     = precon_ice.firn_thickness.domain
precon_odt_space      = precon_odt.height_measure.domain

precon_F   = precon_fp_op
precon_S   = sea_surface_height_operator(precon_fp, precon_response_space)
precon_L_I = precon_ice.ice_thickness_to_load_operator
precon_L_F = precon_ice.firn_thickness_to_load_operator
precon_L_W = sea_level_change_to_load_operator(precon_fp, precon_fp_op.domain)

# Sample at full-resolution coordinates so data spaces match
precon_P_S_ssh  = precon_S.codomain.point_evaluation_operator(ssh_altimetry.coords)
precon_P_S_odt  = precon_odt_space.point_evaluation_operator(ssh_altimetry.coords)
precon_P_I_ice  = precon_ice_space.point_evaluation_operator(ice_altimetry.coords)
precon_P_I_firn = precon_firn_space.point_evaluation_operator(ice_altimetry.coords)
precon_grace_op = grace_operator(precon_response_space, grace_observation_degree)

precon_id_odt  = precon_odt_space.identity_operator()
precon_id_ice  = precon_ice_space.identity_operator()
precon_id_firn = precon_firn_space.identity_operator()

precon_ssh_obs   = precon_P_S_ssh.codomain
precon_ice_obs   = precon_P_I_ice.codomain
precon_grace_obs = precon_grace_op.codomain

precon_L_right = BlockLinearOperator([
    [precon_L_I, precon_L_F, precon_L_W],
    [precon_ice_space.zero_operator(codomain=precon_odt_space),
     precon_firn_space.zero_operator(codomain=precon_odt_space), precon_id_odt],
    [precon_id_ice, precon_firn_space.zero_operator(codomain=precon_ice_space),
     precon_odt_space.zero_operator(codomain=precon_ice_space)],
    [precon_ice_space.zero_operator(codomain=precon_firn_space), precon_id_firn,
     precon_odt_space.zero_operator(codomain=precon_firn_space)],
])
precon_F_middle = BlockDiagonalLinearOperator(
    [precon_F, precon_id_odt, precon_id_ice, precon_id_firn]
)
precon_P_left = BlockLinearOperator([
    [precon_P_S_ssh @ precon_S, precon_P_S_odt,
     precon_ice_space.zero_operator(codomain=precon_ssh_obs),
     precon_firn_space.zero_operator(codomain=precon_ssh_obs)],
    [precon_response_space.zero_operator(codomain=precon_ice_obs),
     precon_odt_space.zero_operator(codomain=precon_ice_obs),
     precon_P_I_ice, precon_P_I_firn],
    [precon_grace_op, precon_odt_space.zero_operator(codomain=precon_grace_obs),
     precon_ice_space.zero_operator(codomain=precon_grace_obs),
     precon_firn_space.zero_operator(codomain=precon_grace_obs)],
])

precon_forward_operator = precon_P_left @ precon_F_middle @ precon_L_right

precon_forward_problem = LinearForwardProblem(
    precon_forward_operator, data_error_measure=data_error_measure,
)
precon_bayesian_inversion = LinearBayesianInversion(
    precon_forward_problem, precon_model_prior
)

print('Forming preconditioner via eigen-decomposition...')
precon_inverse_normal_operator = EigenSolver(parallel=True)(
    precon_bayesian_inversion.normal_operator
)
print('Preconditioner ready.')

### Run the full-resolution inversion

The Bayesian inversion is solved using preconditioned CG, which iteratively
minimises the normal equations:

$$\left( G^T C_\epsilon^{-1} G + C_0^{-1} \right) m = G^T C_\epsilon^{-1} d$$

where $G$ is the forward operator, $C_\epsilon$ is the data error covariance,
$C_0$ is the prior covariance, $m$ is the model, and $d$ is the data vector.

The convergence plot shows $\|x_k\|$ at each CG iteration — a rapid decrease
indicates the preconditioner is working well.

In [ ]:
bayesian_inversion = LinearBayesianInversion(forward_problem, model_prior)

residuals = []
pbar = tqdm(desc='CG solve')

def progress_callback(xk):
    residuals.append(np.linalg.norm(xk))
    pbar.set_postfix({'||x||': f'{residuals[-1]:.2e}'})
    pbar.update(1)

model_posterior_measure = bayesian_inversion.model_posterior_measure(
    data,
    CGMatrixSolver(callback=progress_callback, maxiter=500),
    preconditioner=precon_inverse_normal_operator,
)
pbar.close()
model_posterior_expectation = model_posterior_measure.expectation

fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogy(residuals, marker='o', linestyle='-', markersize=3)
ax.set_title('Convergence of CG solver (GRACE joint inversion)')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$\|x_k\|$')
ax.grid(True, which='both', ls='-', alpha=0.5)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'grace_cg_convergence.pdf', dpi=300)
plt.show()

print(f'CG converged in {len(residuals)} iterations.')

### Posterior component maps

Extract the three model components from the posterior expectation and compare
them to the synthetic truth. Left panels show the truth; right panels show
the posterior mean. All fields are converted to millimetres and masked to
the relevant region (ice or ocean) using the fingerprint projection masks.

In [ ]:
from pyshtools import SHGrid

ice_thickness_true  = model_true[0]
firn_thickness_true = model_true[1]
odt_height_true     = model_true[2]

ice_thickness_posterior  = model_posterior_expectation[0]
firn_thickness_posterior = model_posterior_expectation[1]
odt_height_posterior     = model_posterior_expectation[2]


def plot_shgrid_on_ax(shgrid, ax, *, cmap='seismic', vmin=None, vmax=None):
    data = np.asarray(shgrid.data)
    lons = np.asarray(shgrid.lons())
    lats = np.asarray(shgrid.lats())
    im = ax.pcolormesh(lons, lats, data, transform=ccrs.PlateCarree(),
                       cmap=cmap, shading='auto', vmin=vmin, vmax=vmax)
    ax.coastlines(linewidth=0.6)
    ax.set_global()
    return im


def mm(field):
    return 1000 * field * fp.length_scale


# Ice thickness
ice_proj = fp.ice_projection()
clim_ice = float(np.nanmax(np.abs(np.concatenate([
    (mm(ice_thickness_true) * ice_proj).data.flatten(),
    (mm(ice_thickness_posterior) * ice_proj).data.flatten(),
])))
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5),
                                subplot_kw={'projection': ccrs.Robinson()})
plot_shgrid_on_ax(mm(ice_thickness_true) * ice_proj, ax1, vmin=-clim_ice, vmax=clim_ice)
ax1.set_title('True ice thickness change')
im = plot_shgrid_on_ax(mm(ice_thickness_posterior) * ice_proj, ax2, vmin=-clim_ice, vmax=clim_ice)
ax2.set_title('Posterior expectation (GRACE + altimetry)')
fig.colorbar(im, ax=[ax1, ax2], orientation='vertical', shrink=0.8,
             label='Ice Thickness Change (mm)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'grace_ice_thickness.pdf', dpi=300)
plt.show()

In [ ]:
# Firn thickness
clim_firn = float(np.nanmax(np.abs(np.concatenate([
    (mm(firn_thickness_true) * ice_proj).data.flatten(),
    (mm(firn_thickness_posterior) * ice_proj).data.flatten(),
])))
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5),
                                subplot_kw={'projection': ccrs.Robinson()})
plot_shgrid_on_ax(mm(firn_thickness_true) * ice_proj, ax1, vmin=-clim_firn, vmax=clim_firn)
ax1.set_title('True firn thickness change')
im = plot_shgrid_on_ax(mm(firn_thickness_posterior) * ice_proj, ax2, vmin=-clim_firn, vmax=clim_firn)
ax2.set_title('Posterior expectation')
fig.colorbar(im, ax=[ax1, ax2], orientation='vertical', shrink=0.8,
             label='Firn Thickness Change (mm)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'grace_firn_thickness.pdf', dpi=300)
plt.show()

In [ ]:
# Ocean dynamics height
ocean_proj = fp.ocean_projection()
clim_odt = float(np.nanmax(np.abs(np.concatenate([
    (mm(odt_height_true) * ocean_proj).data.flatten(),
    (mm(odt_height_posterior) * ocean_proj).data.flatten(),
])))
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5),
                                subplot_kw={'projection': ccrs.Robinson()})
plot_shgrid_on_ax(mm(odt_height_true) * ocean_proj, ax1, vmin=-clim_odt, vmax=clim_odt)
ax1.set_title('True ocean dynamics height change')
im = plot_shgrid_on_ax(mm(odt_height_posterior) * ocean_proj, ax2, vmin=-clim_odt, vmax=clim_odt)
ax2.set_title('Posterior expectation')
fig.colorbar(im, ax=[ax1, ax2], orientation='vertical', shrink=0.8,
             label='ODT Height Change (mm)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'grace_odt_height.pdf', dpi=300)
plt.show()

### Joint ice/firn GMSL covariance

One key advantage of combining ice altimetry with GRACE is that their
combination constrains the ice-vs-firn decomposition, reducing the posterior
correlation between ice-GMSL and firn-GMSL contributions.

We compute the 2D joint posterior Gaussian for
$(\Delta\mathrm{GMSL}_{\mathrm{ice}},\, \Delta\mathrm{GMSL}_{\mathrm{firn}})$
using the polarisation identity to extract the cross-covariance without
materialising the full model covariance matrix:

$$\mathrm{Cov}(X, Y) = \tfrac{1}{2}\bigl[\mathrm{Var}(X+Y) - \mathrm{Var}(X) - \mathrm{Var}(Y)\bigr]$$

In [ ]:
ice_gmsl_op  = ice.ice_thickness_to_gmsl_operator
firn_gmsl_op = ice.firn_thickness_to_gmsl_operator

ice_gmsl_row = RowLinearOperator([
    1000 * ice_gmsl_op,
    ice.firn_thickness.domain.zero_operator(codomain=ice_gmsl_op.codomain),
    odt.height_measure.domain.zero_operator(codomain=ice_gmsl_op.codomain),
])
firn_gmsl_row = RowLinearOperator([
    ice.ice_thickness.domain.zero_operator(codomain=firn_gmsl_op.codomain),
    1000 * firn_gmsl_op,
    odt.height_measure.domain.zero_operator(codomain=firn_gmsl_op.codomain),
])

ice_gmsl_post  = model_posterior_measure.affine_mapping(operator=ice_gmsl_row)
firn_gmsl_post = model_posterior_measure.affine_mapping(operator=firn_gmsl_row)
sum_gmsl_post  = model_posterior_measure.affine_mapping(operator=ice_gmsl_row + firn_gmsl_row)

var_ice   = standard_dev(ice_gmsl_post) ** 2
var_firn  = standard_dev(firn_gmsl_post) ** 2
var_sum   = standard_dev(sum_gmsl_post) ** 2
cross_cov = 0.5 * (var_sum - var_ice - var_firn)

mu_ice  = ice_gmsl_post.expectation[0]
mu_firn = firn_gmsl_post.expectation[0]

joint_gmsl_posterior = GaussianMeasure.from_covariance_matrix(
    EuclideanSpace(2),
    np.array([[var_ice, cross_cov], [cross_cov, var_firn]]),
    expectation=np.array([mu_ice, mu_firn])
)

true_ice_gmsl  = ice_gmsl_row(model_true)[0]
true_firn_gmsl = firn_gmsl_row(model_true)[0]

fig_cov, axes_cov = plot_bivariate_corner(
    joint_gmsl_posterior,
    true_values=np.array([true_ice_gmsl, true_firn_gmsl]),
    labels=['Ice GMSL (mm)', 'Firn GMSL (mm)'],
    title='Joint Posterior: Ice vs Firn GMSL Contributions',
    figsize=(6.5, 6.5),
    pdf_colors=[colors.ice, colors.firn],
)
fig_cov.savefig(FIGURES_DIR / 'grace_ice_firn_gmsl_covariance.pdf', dpi=300)
plt.show()

print(f'Ice  GMSL posterior: {mu_ice:.3f} +/- {np.sqrt(var_ice):.3f} mm')
print(f'Firn GMSL posterior: {mu_firn:.3f} +/- {np.sqrt(var_firn):.3f} mm')
print(f'True ice  GMSL: {true_ice_gmsl:.3f} mm')
print(f'True firn GMSL: {true_firn_gmsl:.3f} mm')

---
## Part 2: Knockout Test

To assess the individual contribution of each data type, we run four
inversions from the same synthetic true model with different subsets
of observations:

| Variant | SSH altimetry | Ice altimetry | GRACE |
|---------|:---:|:---:|:---:|
| Full | yes | yes | yes |
| No SSH | | yes | yes |
| No ice altimetry | yes | | yes |
| No GRACE | yes | yes | |

By comparing GMSL posterior distributions across variants we can directly
read off how much each data type contributes to reducing GMSL uncertainty.

The factored forward-operator structure makes this efficient:
$F_{\mathrm{middle}}$ and $L_{\mathrm{right}}$ are shared across all variants;
only $P_{\mathrm{left}}$ changes.

### Build shared operators and variant forward problems

In [ ]:
np.random.seed(42)

load_space     = fp_op.domain
response_space = fp_op.codomain
ice_space      = ice.ice_thickness.domain
firn_space     = ice.firn_thickness.domain
odt_space      = odt.height_measure.domain

F   = fp_op
S   = sea_surface_height_operator(fp, response_space)
L_I = ice.ice_thickness_to_load_operator
L_F = ice.firn_thickness_to_load_operator
L_W = sea_level_change_to_load_operator(fp, load_space)

P_S_ssh  = ssh_altimetry.point_evaluation_operator(S.codomain)
P_S_odt  = ssh_altimetry.point_evaluation_operator(odt_space)
P_I_ice  = ice_altimetry.point_evaluation_operator(ice_space)
P_I_firn = ice_altimetry.point_evaluation_operator(firn_space)
grace_op = grace_operator(response_space, grace_observation_degree)

id_odt  = odt_space.identity_operator()
id_ice  = ice_space.identity_operator()
id_firn = firn_space.identity_operator()

ssh_obs   = P_S_ssh.codomain
ice_obs   = P_I_ice.codomain
grace_obs = grace_op.codomain

# Shared L_right and F_middle
L_right = BlockLinearOperator([
    [L_I, L_F, L_W],
    [ice_space.zero_operator(codomain=odt_space),
     firn_space.zero_operator(codomain=odt_space), id_odt],
    [id_ice, firn_space.zero_operator(codomain=ice_space),
     odt_space.zero_operator(codomain=ice_space)],
    [ice_space.zero_operator(codomain=firn_space), id_firn,
     odt_space.zero_operator(codomain=firn_space)],
])
F_middle = BlockDiagonalLinearOperator([F, id_odt, id_ice, id_firn])

# One row per data type - mix and match to build variants
row_ssh = [P_S_ssh @ S, P_S_odt,
           ice_space.zero_operator(codomain=ssh_obs),
           firn_space.zero_operator(codomain=ssh_obs)]
row_ice = [response_space.zero_operator(codomain=ice_obs),
           odt_space.zero_operator(codomain=ice_obs), P_I_ice, P_I_firn]
row_grace = [grace_op, odt_space.zero_operator(codomain=grace_obs),
             ice_space.zero_operator(codomain=grace_obs),
             firn_space.zero_operator(codomain=grace_obs)]

err_ssh   = GaussianMeasure.from_standard_deviation(ssh_obs,   measure_error_std)
err_ice   = GaussianMeasure.from_standard_deviation(ice_obs,   measure_error_std)
err_grace = GaussianMeasure.from_standard_deviation(grace_obs, grace_std)


def build_variant(selected_rows, selected_errors):
    p_left     = BlockLinearOperator(selected_rows)
    forward_op = p_left @ F_middle @ L_right
    data_error = GaussianMeasure.from_direct_sum(selected_errors)
    return forward_op, data_error


forward_op_full,     data_error_full     = build_variant([row_ssh, row_ice, row_grace], [err_ssh, err_ice, err_grace])
forward_op_no_ssh,   data_error_no_ssh   = build_variant([row_ice, row_grace],          [err_ice, err_grace])
forward_op_no_ice,   data_error_no_ice   = build_variant([row_ssh, row_grace],          [err_ssh, err_grace])
forward_op_no_grace, data_error_no_grace = build_variant([row_ssh, row_ice],            [err_ssh, err_ice])

print('Variant forward operators built.')

### Generate synthetic data for all variants

Sample a single `model_true` from the full problem's prior, then apply each
variant's forward operator and add independent noise. All variants share
the same underlying truth.

In [ ]:
full_problem_for_sampling = LinearForwardProblem(
    forward_op_full, data_error_measure=data_error_full
)
model_true_ko, data_full_ko = (
    full_problem_for_sampling.synthetic_model_and_data(model_prior)
)

def make_data(forward_op, data_error):
    return forward_op(model_true_ko) + data_error.sample()

data_no_ssh   = make_data(forward_op_no_ssh,   data_error_no_ssh)
data_no_ice   = make_data(forward_op_no_ice,   data_error_no_ice)
data_no_grace = make_data(forward_op_no_grace, data_error_no_grace)

print('Synthetic data for all knockout variants generated.')

### Build preconditioners for each variant

Each variant needs its own preconditioner because removing observation rows
changes the structure of the normal operator. We reuse the low-resolution
fingerprint from Part 1 and build a separate eigen-decomposed approximate
inverse for each variant.

In [ ]:
precon_row_ssh = [
    precon_P_S_ssh @ precon_S, precon_P_S_odt,
    precon_ice_space.zero_operator(codomain=precon_P_S_ssh.codomain),
    precon_firn_space.zero_operator(codomain=precon_P_S_ssh.codomain),
]
precon_row_ice = [
    precon_response_space.zero_operator(codomain=precon_P_I_ice.codomain),
    precon_odt_space.zero_operator(codomain=precon_P_I_ice.codomain),
    precon_P_I_ice, precon_P_I_firn,
]
precon_row_grace = [
    precon_grace_op, precon_odt_space.zero_operator(codomain=precon_grace_op.codomain),
    precon_ice_space.zero_operator(codomain=precon_grace_op.codomain),
    precon_firn_space.zero_operator(codomain=precon_grace_op.codomain),
]


def build_preconditioner(selected_precon_rows, full_res_data_error, label):
    p_left         = BlockLinearOperator(selected_precon_rows)
    precon_fwd_op  = p_left @ precon_F_middle @ precon_L_right
    precon_problem = LinearForwardProblem(
        precon_fwd_op, data_error_measure=full_res_data_error
    )
    precon_inv_obj = LinearBayesianInversion(precon_problem, precon_model_prior)
    print(f'  Building preconditioner: {label}...')
    inv_normal = EigenSolver(parallel=False)(precon_inv_obj.normal_operator)
    print(f'  Done: {label}')
    return inv_normal


precon_inv_full     = build_preconditioner([precon_row_ssh, precon_row_ice, precon_row_grace], data_error_full,     'full')
precon_inv_no_ssh   = build_preconditioner([precon_row_ice, precon_row_grace],                  data_error_no_ssh,   'no SSH')
precon_inv_no_ice   = build_preconditioner([precon_row_ssh, precon_row_grace],                  data_error_no_ice,   'no ice')
precon_inv_no_grace = build_preconditioner([precon_row_ssh, precon_row_ice],                    data_error_no_grace, 'no GRACE')

### Run all knockout inversions

Run all four inversions in parallel using `joblib`. For each posterior we compute
the total GMSL posterior (mean and std in mm) and the 2D joint posterior for
(ice GMSL, firn GMSL) using the polarisation identity for the cross-covariance.

In [ ]:
from joblib import Parallel, delayed

ice_gmsl_op  = ice.ice_thickness_to_gmsl_operator
firn_gmsl_op = ice.firn_thickness_to_gmsl_operator

odt_zero_op   = odt.height_measure.domain.zero_operator(codomain=ice_gmsl_op.codomain)
total_gmsl_op = RowLinearOperator([ice_gmsl_op, firn_gmsl_op, odt_zero_op])
total_gmsl_true_mm = total_gmsl_op(model_true_ko)[0] * 1000

ko_ice_gmsl_row = RowLinearOperator([
    1000 * ice_gmsl_op,
    ice.firn_thickness.domain.zero_operator(codomain=ice_gmsl_op.codomain),
    odt.height_measure.domain.zero_operator(codomain=ice_gmsl_op.codomain),
])
ko_firn_gmsl_row = RowLinearOperator([
    ice.ice_thickness.domain.zero_operator(codomain=firn_gmsl_op.codomain),
    1000 * firn_gmsl_op,
    odt.height_measure.domain.zero_operator(codomain=firn_gmsl_op.codomain),
])


def run_inversion(forward_op, data_error, data, precon_inv, label):
    problem   = LinearForwardProblem(forward_op, data_error_measure=data_error)
    inversion = LinearBayesianInversion(problem, model_prior)
    residuals = []
    pbar      = tqdm(desc=f'CG ({label})')
    solving   = [True]

    def callback(xk):
        if solving[0]:
            residuals.append(np.linalg.norm(xk))
            pbar.set_postfix({'||x||': f'{residuals[-1]:.2e}'})
            pbar.update(1)

    posterior = inversion.model_posterior_measure(
        data, CGMatrixSolver(callback=callback, maxiter=500, rtol=1e-5),
        preconditioner=precon_inv,
    )
    pbar.close()
    solving[0] = False
    return posterior, residuals


def compute_gmsl_posterior(posterior):
    post_measure = posterior.affine_mapping(operator=total_gmsl_op)
    exp_mm = post_measure.expectation[0] * 1000
    var    = float(post_measure.covariance.matrix(dense=True, parallel=False)[0, 0])
    return exp_mm, np.sqrt(max(var, 0.0)) * 1000


def gmsl_2d_posterior(posterior):
    ice_post  = posterior.affine_mapping(operator=ko_ice_gmsl_row)
    firn_post = posterior.affine_mapping(operator=ko_firn_gmsl_row)
    sum_post  = posterior.affine_mapping(operator=ko_ice_gmsl_row + ko_firn_gmsl_row)
    var_ice   = standard_dev(ice_post,  parallel=False) ** 2
    var_firn  = standard_dev(firn_post, parallel=False) ** 2
    var_sum   = standard_dev(sum_post,  parallel=False) ** 2
    cross_cov = 0.5 * (var_sum - var_ice - var_firn)
    mu = np.array([ice_post.expectation[0], firn_post.expectation[0]])
    return GaussianMeasure.from_covariance_matrix(
        EuclideanSpace(2),
        np.array([[var_ice, cross_cov], [cross_cov, var_firn]]),
        expectation=mu,
    )


def run_task(args):
    posterior, residuals = run_inversion(*args)
    return posterior, residuals, compute_gmsl_posterior(posterior), gmsl_2d_posterior(posterior)


tasks = [
    (forward_op_full,     data_error_full,     data_full_ko,   precon_inv_full,     'full'),
    (forward_op_no_ssh,   data_error_no_ssh,   data_no_ssh,    precon_inv_no_ssh,   'no SSH'),
    (forward_op_no_ice,   data_error_no_ice,   data_no_ice,    precon_inv_no_ice,   'no ice'),
    (forward_op_no_grace, data_error_no_grace, data_no_grace,  precon_inv_no_grace, 'no GRACE'),
]

print('Running all knockout inversions in parallel...')
results = Parallel(n_jobs=-1, backend='multiprocessing')(
    delayed(run_task)(args) for args in tasks
)

(
    (posterior_full,     res_full,     gmsl_full,     post2d_full),
    (posterior_no_ssh,   res_no_ssh,   gmsl_no_ssh,   post2d_no_ssh),
    (posterior_no_ice,   res_no_ice,   gmsl_no_ice,   post2d_no_ice),
    (posterior_no_grace, res_no_grace, gmsl_no_grace, post2d_no_grace),
) = results

print('All inversions complete.')

### CG convergence comparison

Plot the CG residual norm for all four variants. Variants with fewer
observations typically converge faster because the problem is less
constrained and the preconditioner is a closer match to the true inverse.

In [ ]:
variant_residuals = [
    ('Full (SSH + ice + GRACE)', res_full,     colors.new_method),
    ('No SSH altimetry',         res_no_ssh,   colors.ice_altimetry),
    ('No ice altimetry',         res_no_ice,   colors.ocean_altimetry),
    ('No GRACE',                 res_no_grace, colors.ocean_dynamics),
]

fig, ax = plt.subplots(figsize=(8, 4))
for label, residuals, color in variant_residuals:
    ax.semilogy(residuals, label=label, color=color, linewidth=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$\|x_k\|$')
ax.set_title('CG convergence - knockout variants')
ax.legend(fontsize=8)
ax.grid(True, which='both', ls='-', alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'knockout_cg_convergence.pdf', dpi=300)
plt.show()

### GMSL posterior comparison

Compare the total GMSL (ice + firn) posterior for each variant. A narrower
distribution means lower GMSL uncertainty. Removing an informative data
type widens the posterior.

In [ ]:
from scipy import stats

variant_gmsl = [
    ('Full (SSH + ice + GRACE)', gmsl_full,     colors.new_method),
    ('No SSH altimetry',         gmsl_no_ssh,   colors.ice_altimetry),
    ('No ice altimetry',         gmsl_no_ice,   colors.ocean_altimetry),
    ('No GRACE',                 gmsl_no_grace, colors.firn),
]

finite_stds = [s for _, (_, s), _ in variant_gmsl if s > 1e-6]
x_half  = 4 * max(finite_stds) if finite_stds else 5.0
x_range = np.linspace(total_gmsl_true_mm - x_half, total_gmsl_true_mm + x_half, 1000)

fig, ax = plt.subplots(figsize=(9, 4))
ax.axvline(total_gmsl_true_mm, color=colors.true, linestyle='--', linewidth=2,
           label=f'True GMSL ({total_gmsl_true_mm:.2f} mm)')

for label, (exp_mm, std_mm), color in variant_gmsl:
    pdf = stats.norm.pdf(x_range, exp_mm, std_mm)
    ax.plot(x_range, pdf, color=color, linewidth=1.8,
            label=f'{label}\n(mean={exp_mm:.2f}, std={std_mm:.2e} mm)')
    ax.axvline(exp_mm, color=color, linestyle=':', linewidth=1, alpha=0.6)

ax.get_yaxis().set_visible(False)
ax.set_xlabel('GMSL Contribution (mm)')
ax.set_title('Knockout test: GMSL posterior distributions')
ax.legend(fontsize=8, loc='upper left')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'knockout_gmsl.pdf', dpi=300)
plt.show()

print(f'True GMSL: {total_gmsl_true_mm:.4f} mm')
for label, (exp_mm, std_mm), _ in variant_gmsl:
    sigma = abs(exp_mm - total_gmsl_true_mm) / std_mm if std_mm > 1e-6 else float('inf')
    print(f'  {label:<30}: mean={exp_mm:.4f} mm, std={std_mm:.2e} mm, {sigma:.2f} sigma from truth')

### Component grid: true vs posterior expectations

For each of the three model components (ice thickness, firn thickness, ocean
dynamics height), plot the truth alongside the posterior expectation for all
four variants. This reveals which components benefit most from each data type:

- **GRACE** primarily constrains *total ice+firn mass*, so removing it degrades
  the ability to disentangle ice from firn.
- **Ice altimetry** constrains the *spatial pattern* of thickness change on the ice sheets.
- **SSH altimetry** constrains the ocean dynamics component and the fingerprint shape.

In [ ]:
posteriors_ordered = [
    ('Full\n(SSH+ice+GRACE)', posterior_full),
    ('No SSH\naltimetry',     posterior_no_ssh),
    ('No ice\naltimetry',     posterior_no_ice),
    ('No GRACE',              posterior_no_grace),
]

ice_true_ko  = model_true_ko[0]
firn_true_ko = model_true_ko[1]
odt_true_ko  = model_true_ko[2]


def field_mm(shgrid, projection_mask):
    return (shgrid * projection_mask * fp.length_scale * 1000).data.astype(float)


def sym_clim(*arrays):
    vals = np.concatenate([a[np.isfinite(a)].ravel() for a in arrays])
    return np.nanmax(np.abs(vals))


component_rows = []
for comp_key, proj, unit_label in [
    ('ice',  fp.ice_projection(),   'Ice Thickness Change (mm)'),
    ('firn', fp.ice_projection(),   'Firn Thickness Change (mm)'),
    ('odt',  fp.ocean_projection(), 'Ocean Dyn. Height (mm)'),
]:
    true_field = {'ice': ice_true_ko, 'firn': firn_true_ko, 'odt': odt_true_ko}[comp_key]
    idx        = {'ice': 0, 'firn': 1, 'odt': 2}[comp_key]
    row_arrays = [field_mm(true_field, proj)]
    for _, post in posteriors_ordered:
        row_arrays.append(field_mm(post.expectation[idx], proj))
    component_rows.append((comp_key, unit_label, row_arrays, sym_clim(*row_arrays)))

_raw_lats     = ice_true_ko.lats()
_raw_lons     = ice_true_ko.lons()
_lons_shifted = np.where(_raw_lons > 180, _raw_lons - 360, _raw_lons)
_sort_idx     = np.argsort(_lons_shifted)
_lon_grid, _lat_grid = np.meshgrid(_lons_shifted[_sort_idx], _raw_lats)

n_cols = 1 + len(posteriors_ordered)
fig_grid, axes = plt.subplots(
    3, n_cols, figsize=(12, 7),
    subplot_kw={'projection': ccrs.Robinson()},
    constrained_layout=True,
)
col_titles = ['True'] + [lbl for lbl, _ in posteriors_ordered]

for row_idx, (comp_key, unit_label, row_arrays, clim) in enumerate(component_rows):
    for col_idx, (arr, col_title) in enumerate(zip(row_arrays, col_titles)):
        ax = axes[row_idx, col_idx]
        im = ax.pcolormesh(_lon_grid, _lat_grid, arr[:, _sort_idx],
                           transform=ccrs.PlateCarree(),
                           cmap='seismic', vmin=-clim, vmax=clim)
        ax.coastlines(linewidth=0.4, color='k')
        if row_idx == 0:
            ax.set_title(col_title, fontsize=7, pad=4)
        if col_idx == 0:
            ax.text(-0.04, 0.5, unit_label, va='center', ha='right',
                    rotation=90, fontsize=7.5, transform=ax.transAxes)
    cb = fig_grid.colorbar(im, ax=axes[row_idx, :],
                            orientation='horizontal', shrink=0.6, label=unit_label)
    cb.ax.tick_params(labelsize=7)

fig_grid.suptitle(
    'Knockout test: true vs posterior expectations\n'
    '(rows: ice / firn / ocean dynamics; columns: true and each inversion variant)',
    fontsize=9,
)
fig_grid.savefig(FIGURES_DIR / 'knockout_component_grid.pdf', dpi=300, bbox_inches='tight')
plt.show()

### Bivariate GMSL overlay

Overlay the 2D joint posterior (ice GMSL, firn GMSL) for all four variants.
The 1-sigma (solid) and 2-sigma (dotted) contours show how joint uncertainty
changes as data types are removed.

Key things to look for:
- **Orientation** — does removing a data type tilt the ellipses, introducing
  a correlation between ice and firn GMSL?
- **Size** — which data type most reduces uncertainty in the joint GMSL plane?

In [ ]:
variants_2d = [
    ('Full (SSH+ice+GRACE)', post2d_full,     colors.new_method),
    ('No SSH altimetry',     post2d_no_ssh,   colors.ice_altimetry),
    ('No ice altimetry',     post2d_no_ice,   colors.ocean_altimetry),
    ('No GRACE',             post2d_no_grace, colors.firn),
]

true_ice_gmsl_mm_ko  = ko_ice_gmsl_row(model_true_ko)[0]
true_firn_gmsl_mm_ko = ko_firn_gmsl_row(model_true_ko)[0]

fig_ov, axes_ov = plt.subplots(
    2, 2, figsize=(8, 8),
    gridspec_kw={'width_ratios': [2, 1], 'height_ratios': [1, 2]},
)
ax_top    = axes_ov[0, 0]
ax_main   = axes_ov[1, 0]
ax_right  = axes_ov[1, 1]
ax_legend = axes_ov[0, 1]
ax_legend.axis('off')

for label, measure_2d, color in variants_2d:
    mu  = measure_2d.expectation
    cov = measure_2d.covariance.matrix(dense=True, parallel=False)
    sigma0 = np.sqrt(cov[0, 0])
    sigma1 = np.sqrt(cov[1, 1])

    x0 = np.linspace(mu[0] - 4*sigma0, mu[0] + 4*sigma0, 300)
    ax_top.plot(x0, stats.norm.pdf(x0, mu[0], sigma0),
                color=color, linewidth=1.6, label=label)

    x1 = np.linspace(mu[1] - 4*sigma1, mu[1] + 4*sigma1, 300)
    ax_right.plot(stats.norm.pdf(x1, mu[1], sigma1), x1, color=color, linewidth=1.6)

    rv      = stats.multivariate_normal(mu, cov)
    level_1 = rv.pdf(mu) * np.exp(-0.5)
    level_2 = rv.pdf(mu) * np.exp(-2.0)
    xg = np.linspace(mu[0] - 3.75*sigma0, mu[0] + 3.75*sigma0, 120)
    yg = np.linspace(mu[1] - 3.75*sigma1, mu[1] + 3.75*sigma1, 120)
    X, Y = np.meshgrid(xg, yg)
    Z = rv.pdf(np.dstack((X, Y)))
    ax_main.contour(X, Y, Z, levels=[level_1], colors=[color], linewidths=1.8, linestyles='-')
    ax_main.contour(X, Y, Z, levels=[level_2], colors=[color], linewidths=1.8, linestyles=':')
    ax_main.plot(mu[0], mu[1], '+', color=color, markersize=8, mew=2)

ax_top.axvline(true_ice_gmsl_mm_ko,  color=colors.true, linestyle='--', linewidth=1.5, label='True')
ax_right.axhline(true_firn_gmsl_mm_ko, color=colors.true, linestyle='--', linewidth=1.5)
ax_main.plot(true_ice_gmsl_mm_ko, true_firn_gmsl_mm_ko, 'kx',
             markersize=10, mew=2, label='True', zorder=5)

ax_top.set_ylabel('Density')
ax_top.set_xticklabels([])
ax_top.set_yticklabels([])
ax_right.set_xlabel('Density')
ax_right.set_yticklabels([])
ax_main.set_xlabel('Ice GMSL (mm)')
ax_main.set_ylabel('Firn GMSL (mm)')

handles, labels_leg = ax_top.get_legend_handles_labels()
h_main, l_main     = ax_main.get_legend_handles_labels()
ax_legend.legend(handles + h_main, labels_leg + l_main,
                 loc='center', fontsize=9, frameon=False)

fig_ov.suptitle(
    'Knockout sensitivity: Ice vs Firn GMSL\n'
    '(1-sigma solid, 2-sigma dotted; all variants)',
    fontsize=12,
)
plt.tight_layout()
fig_ov.savefig(FIGURES_DIR / 'knockout_gmsl_bivariate_overlay.pdf', dpi=300)
plt.show()